In [1]:
!pip install ultralytics opencv-python


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import cv2
from ultralytics import YOLO

# Load YOLO Model
model = YOLO("yolov8n.pt")

# Input & Output Folders
input_folder = "videoo"
output_folder = "output_videos"

os.makedirs(output_folder, exist_ok=True)

# Get all video files
video_files = [
    f for f in os.listdir(input_folder)
    if f.lower().endswith((".mp4", ".avi", ".mov", ".mkv"))
]

print(f"Found {len(video_files)} video(s).\n")

# Process Videos
for video_name in video_files:

    print(f"Processing: {video_name}")

    video_path = os.path.join(input_folder, video_name)

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print("Could not open video.\n")
        continue

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps == 0:
        fps = 30

    output_path = os.path.join(output_folder, video_name)

    writer = cv2.VideoWriter(
        output_path,
        cv2.VideoWriter_fourcc(*'mp4v'),
        fps,
        (width, height)
    )

    # Store unique IDs
    unique_ids = set()

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        # Object Tracking
        results = model.track(
            frame,
            persist=True,
            tracker="bytetrack.yaml",
            verbose=False
        )

        result = results[0]

        # Save tracking IDs
        if result.boxes.id is not None:

            ids = result.boxes.id.cpu().numpy().astype(int)

            for object_id in ids:
                unique_ids.add(object_id)

        # Draw bounding boxes + IDs
        annotated_frame = result.plot()

        # Save frame
        writer.write(annotated_frame)

    cap.release()
    writer.release()

    print(f"Unique Objects Detected : {len(unique_ids)}")
    print(f"Saved Output            : {output_path}")
    print("-" * 50)

print("\nAll videos processed successfully!")

Found 5 video(s).

Processing: video1.mp4
Unique Objects Detected : 60
Saved Output            : output_videos\video1.mp4
--------------------------------------------------
Processing: video2.mp4
Unique Objects Detected : 323
Saved Output            : output_videos\video2.mp4
--------------------------------------------------
Processing: video3.mp4
Unique Objects Detected : 38
Saved Output            : output_videos\video3.mp4
--------------------------------------------------
Processing: video4.mp4
Unique Objects Detected : 6
Saved Output            : output_videos\video4.mp4
--------------------------------------------------
Processing: video7.mp4
Unique Objects Detected : 11
Saved Output            : output_videos\video7.mp4
--------------------------------------------------

All videos processed successfully!
